# Optional Extension: Tiny Local RAG Demo

## Learning goals
- Understand the core idea behind retrieval-augmented generation (RAG).
- Encode tiny local documents into embeddings.
- Retrieve relevant chunks for a query.
- Compare a generic answer with a grounded answer.

We choose a lightweight RAG demo here because it is usually more reliable and easier to teach in a short Colab workshop than a tiny LoRA fine-tuning example.


In [ ]:
# Notebook-specific environment setup.
# We install missing packages so the notebook can run directly in Colab.

import importlib.util
import subprocess
import sys


REQUIRED_PACKAGES = {
    "sentence_transformers": "sentence-transformers>=3.0.0",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
}

for import_name, pip_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"{import_name} is already installed.")


## What is RAG?

RAG stands for **retrieval-augmented generation**.

A simple version looks like this:
1. turn documents into embeddings
2. embed the user query
3. retrieve the most similar chunks
4. answer using those retrieved chunks

**Why this matters**

RAG is often useful when knowledge is local, recent, community-specific, or too dynamic to hard-code into a model.


In [ ]:
# Import the tools we need and define a tiny local document set.
# These short notes are intentionally small so the concept stays easy to follow.

import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


documents = [
    {
        "source": "Program overview",
        "text": "The Riverbend Community Health program offers free digital wellness coaching for adults with diabetes or prediabetes.",
    },
    {
        "source": "Schedule",
        "text": "Evening sessions run on Tuesdays and Thursdays from 5:30 PM to 7:00 PM, which helps people attend after work.",
    },
    {
        "source": "Languages",
        "text": "Coaching is available in English and French, and interpreters can be arranged with advance notice.",
    },
    {
        "source": "Privacy",
        "text": "Participant notes stay on the clinic's secure local system and are not sent to outside advertising platforms.",
    },
    {
        "source": "Eligibility",
        "text": "The program is open to Riverbend residents aged 18 and older. A referral is helpful but not required.",
    },
    {
        "source": "Technology support",
        "text": "Borrower tablets are available for participants who do not have a suitable device at home.",
    },
    {
        "source": "Follow-up",
        "text": "After the first month, participants receive a short progress check-in every two weeks.",
    },
    {
        "source": "Limitations",
        "text": "This service does not replace emergency care, medication prescribing, or in-person diagnosis.",
    },
]

documents_df = pd.DataFrame(documents)
display(documents_df)


## Why we start with notebook-defined documents

This avoids external databases and keeps the demo focused on the core idea.
Later, the same retrieval pattern could connect to a larger local document collection.


In [ ]:
# Load a lightweight sentence embedding model and encode all document chunks.
# The first run may take a moment because the embedding model may need to download.

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

document_texts = [doc["text"] for doc in documents]
document_embeddings = embedding_model.encode(document_texts, convert_to_numpy=True)

print(f"Number of document chunks: {len(document_texts)}")
print(f"Embedding shape for one chunk: {document_embeddings[0].shape}")


## Important embedding lines

- `SentenceTransformer(...)`: loads a pretrained embedding model.
- `document_texts = [doc["text"] for doc in documents]`: extracts only the text we want to embed.
- `embedding_model.encode(..., convert_to_numpy=True)`: converts each text chunk into a numeric vector.

The embedding vectors are what make semantic retrieval possible.


In [ ]:
# Retrieve the top-k most similar chunks for a given query.
# Cosine similarity is a common first step for embedding search.

def retrieve(query, top_k=3):
    # Encode the query into the same vector space as the documents.
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)

    # Compare the query vector with every document vector.
    similarities = cosine_similarity(query_embedding, document_embeddings)[0]

    # Sort from most similar to least similar and keep the top-k indices.
    ranked_indices = similarities.argsort()[::-1][:top_k]

    rows = []
    for index in ranked_indices:
        rows.append(
            {
                "source": documents[index]["source"],
                "text": documents[index]["text"],
                "similarity": similarities[index],
            }
        )

    return pd.DataFrame(rows)


query = "Can I attend after work, and how is my information stored?"
retrieved_df = retrieve(query, top_k=3)

print(f"Query: {query}")
display(retrieved_df.style.format({"similarity": "{:.3f}"}))


## What the retrieval function does

- `query_embedding = embedding_model.encode([query], ...)`: turns the user question into a vector.
- `cosine_similarity(...)`: measures how similar the query vector is to each document vector.
- `argsort()[::-1][:top_k]`: ranks results from most similar to least similar.
- The returned DataFrame makes it easy to inspect both the text and the similarity score.


In [ ]:
# Build two answer styles:
# 1. a generic answer without retrieval
# 2. a grounded answer that references retrieved notes

def answer_without_retrieval(query):
    return (
        "A generic assistant might answer in broad terms, "
        "but it would not know the local program details with confidence."
    )


def answer_with_retrieval(query, top_k=3, min_similarity=0.25):
    top_chunks = retrieve(query, top_k=top_k)
    strong_chunks = top_chunks[top_chunks["similarity"] >= min_similarity]

    if strong_chunks.empty:
        return "I do not have enough support in the local notes to answer this confidently."

    evidence_lines = [
        f"- {row.text} ({row.source})"
        for row in strong_chunks.itertuples(index=False)
    ]

    return "Grounded answer based on the retrieved notes:\n" + "\n".join(evidence_lines)


print("Without retrieval:")
print(answer_without_retrieval(query))

print("\nWith retrieval:")
print(answer_with_retrieval(query))


## Why this RAG pattern is useful

Retrieval gives the answer a local evidence base.
That can help with:
- traceability
- current information
- community-specific knowledge
- privacy and local control


In [ ]:
# Experiment with several example queries.
# This gives learners a quick way to see which kinds of questions retrieve useful evidence.

example_queries = [
    "Do I need a referral to join?",
    "Can I get help if I do not have a device at home?",
    "Does this replace emergency care?",
    "Is coaching available in Spanish?",
]

for demo_query in example_queries:
    print("=" * 90)
    print(f"Query: {demo_query}")
    display(retrieve(demo_query, top_k=2).style.format({"similarity": "{:.3f}"}))
    print(answer_with_retrieval(demo_query, top_k=2))


## Exercise

Edit the next cell to experiment.

Ideas:
- add a new local document
- ask a question the notes cannot answer
- change `top_k`
- raise or lower `min_similarity`
- discuss how this could help with community-specific knowledge


In [ ]:
# Add one extra local note, rebuild embeddings, and try a new query.
# This shows how easy it is to update a tiny retrieval system.

extended_documents = documents + [
    {
        "source": "Transportation",
        "text": "Bus tickets can be provided for participants who need help traveling to the clinic.",
    }
]

extended_texts = [doc["text"] for doc in extended_documents]
extended_embeddings = embedding_model.encode(extended_texts, convert_to_numpy=True)


def retrieve_from_extended_docs(query, top_k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    similarities = cosine_similarity(query_embedding, extended_embeddings)[0]
    ranked_indices = similarities.argsort()[::-1][:top_k]

    rows = []
    for index in ranked_indices:
        rows.append(
            {
                "source": extended_documents[index]["source"],
                "text": extended_documents[index]["text"],
                "similarity": similarities[index],
            }
        )

    return pd.DataFrame(rows)


new_query = "Is there any transportation help for participants?"
display(retrieve_from_extended_docs(new_query).style.format({"similarity": "{:.3f}"}))


## Discussion prompts and recap

- When would RAG be better than fine-tuning a model?
- What happens if the local documents are outdated or incomplete?
- When might local deployment matter for privacy or governance?
- What should we evaluate besides whether the answer sounds fluent?

In this extension notebook, you built a tiny retrieval workflow with local documents, embeddings, semantic search, and grounded answers.
